In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from sklearn.tree import export_graphviz
from sklearn.metrics import classification_report
from sklearn.tree import plot_tree 
from IPython.display import Image
from sklearn import tree

In [ ]:
# 设置全局字体为黑体
plt.rcParams['font.family'] = 'SimHei'
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
female_data = pd.read_excel('附件.xlsx', sheet_name='女胎检测数据')

In [ ]:
female_data.isna().sum()

In [ ]:
del female_data['Unnamed: 20']
del female_data['Unnamed: 21']

In [ ]:
female_data['孕妇BMI'] = female_data['孕妇BMI'].fillna(female_data['体重']/(female_data['身高']/100)**2)

In [ ]:
#仅保留自然受孕的，删去人工授精
female_data = female_data[female_data['IVF妊娠']=='自然受孕']

In [ ]:
plt.figure(figsize=(8,7))
sns.boxplot(female_data[['13号染色体的Z值','18号染色体的Z值','21号染色体的Z值','X染色体的Z值']])

In [ ]:
sns.boxplot(female_data['孕妇BMI'])

In [ ]:
sns.boxplot(female_data['X染色体浓度'])

In [ ]:
#根据3sigma原则删去偏离值
female_data = female_data[(-3<female_data['13号染色体的Z值']) & (female_data['13号染色体的Z值']<3) ]
female_data = female_data[(-3<female_data['18号染色体的Z值']) & (female_data['18号染色体的Z值']<3) ]
female_data = female_data[(-3<female_data['21号染色体的Z值']) & (female_data['21号染色体的Z值']<3) ]
female_data = female_data[(-3<female_data['X染色体的Z值']) & (female_data['X染色体的Z值']<3) ]

In [ ]:
#根据1.5IQR原则删去BMI异常值
BMI_Q1 = female_data['孕妇BMI'].quantile(0.25)
BMI_Q3 = female_data['孕妇BMI'].quantile(0.75)
BMI_IQR = BMI_Q3-BMI_Q1
BMI_lower = BMI_Q1 - 1.5 * BMI_IQR
BMI_upper = BMI_Q3 + 1.5 * BMI_IQR
female_data = female_data[(BMI_lower<female_data['孕妇BMI']) & (female_data['孕妇BMI']<BMI_upper) ]

In [ ]:
#根据1.5IQR原则删去X染色体浓度异常值
X_Q1 = female_data['X染色体浓度'].quantile(0.25)
X_Q3 = female_data['X染色体浓度'].quantile(0.75)
X_IQR = X_Q3-X_Q1
X_lower = X_Q1 - 1.5 * X_IQR
X_upper = X_Q3 + 1.5 * X_IQR
female_data = female_data[(X_lower<female_data['X染色体浓度']) & (female_data['X染色体浓度']<X_upper) ]

In [ ]:
#序号重新编排
female_data = female_data.reset_index()

In [ ]:
#将孕周格式统一为一个小数
def transform_yunzhou(data):
    i = 0
    weeks = 0
    while data[i]!='w' and data[i]!='W':
        i+=1
    weeks += int(data[:i])
    if data[-1]=='w' or data[-1]=='W':
        return weeks
    if data[i+1]=='+':
        weeks += int(data[i+2])/7
    return weeks

In [ ]:
female_data['检测孕周'] = female_data['检测孕周'].apply(lambda x:transform_yunzhou(x))

In [ ]:
female_data

In [ ]:
female_data['是否异常'] = female_data['染色体的非整倍体'].apply(lambda x: 0 if pd.isnull(x) else 1)

In [ ]:
y = female_data['是否异常']
target = [ '年龄',  '检测孕周', '孕妇BMI', '原始读段数', '在参考基因组上比对的比例', '重复读段的比例',
       '唯一比对的读段数', 'GC含量', '13号染色体的Z值', '18号染色体的Z值', '21号染色体的Z值', 'X染色体的Z值',
       'X染色体浓度', '13号染色体的GC含量', '18号染色体的GC含量', '21号染色体的GC含量', '被过滤掉读段数的比例']
X = female_data[target]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
clf = DecisionTreeClassifier(max_depth=3)
clf.fit(X_train, y_train)

In [ ]:
y_pred = clf.predict(X_test)

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy}")

In [ ]:
print(classification_report(y_test, y_pred))

In [ ]:
plt.figure(figsize=(20, 10))
tree.plot_tree(
    clf, 
    feature_names=target,
    class_names=['正常', '异常'],
    filled=True,
    rounded=True,
    fontsize=10
)
plt.title("决策树可视化")
plt.show()

In [ ]:
from imblearn.over_sampling import SMOTE
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report

# 使用SMOTE进行过采样
smote = SMOTE(random_state=42,k_neighbors=2,sampling_strategy=0.5)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print("SMOTE过采样后类别分布:", np.bincount(y_train_smote))

# 训练决策树
dt_clf = DecisionTreeClassifier(
    max_depth=3,
    random_state=42
)

dt_clf.fit(X_train_smote, y_train_smote)
y_pred = dt_clf.predict(X_test)

print("SMOTE过采样 + 决策树 性能报告:")
print(classification_report(y_test, y_pred))

In [ ]:
plt.figure(figsize=(20, 10))
tree.plot_tree(
    dt_clf, 
    feature_names=target,
    class_names=['正常', '异常'],
    filled=True,
    rounded=True,
    fontsize=10
)
plt.title("决策树可视化")
plt.show()